# Notebook for calculating the integrated change of length data for Cuk+21

## Preamble

### Load required modules

In [1]:

import numpy as np
import scipy as sp
import sys
import os
import struct
from scipy import constants as const

import h5py

import time as tclock

from scipy.signal import savgol_filter

#package to use wildcards 
import fnmatch

import csv

from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import griddata

from scipy import interpolate

#plotting packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import pylab
import matplotlib.cm as cm
from matplotlib import gridspec


cwd = os.getcwd()
print(cwd)
if sys.platform== 'darwin':
    sys.path.insert(0, cwd+'/Support_scipts')
    print(cwd+'/Support_scripts')
elif (sys.platform== 'win32') | (sys.platform== 'win64'):
    sys.path.insert(0,cwd+"\\Support_scipts")
    
#HERCULES_structures
from HERCULES_structures import *
from surface_size_calc import *
from HERCULES_random_planet_database_structure_1D import *

#functions for calculating non-evenly spaced numerical differentials
from gradients import *

#import colormaps
import colormaps as cmaps
import matplotlib.cm as cm

import svglib.svglib as svglib
svglib.register_font('helvetica', './Helvetica.ttc')

/Users/vq21447/Documents/Lock_2026_SI
/Users/vq21447/Documents/Lock_2026_SI/Support_scripts
CHECK THE LOCATION OF ODYSSEY BACKUP
CHECK THE LOCATION OF ODYSSEY BACKUP


('helvetica', True)

### Required constants

In [2]:

#CONSTANTS
MEarth=5.972E24
LEM=3.5E34
REarth=6.371E6
MMoon=7.34767309E22

aMoon=0.3844E9
aCassini=30*REarth
aRoche=2.9*REarth

#for HERCULES
MEarth_H=5.9879648E24
LEM_H=3.53E34

### Parameters

In [3]:
#PARAMS
#info for HERCULES arrays
Hdir='Earth_correct_params_S3.20c'
Hname='Earth_correct_params_S3.20c'
    
#directory for max change of length data
data_dir='Data/Cuk_et_al_2021_high_obliquity'

#whether to overwrite saved data or make a new file
flag_overwrite=0

#which plot do you want
#0: Figure 3
#1: Figure 7
flag_data=0

#Earth's moment of inertia used to convert to AM
C_Earth=0.3304

#whether to print the full surface change arrays rather than just integrated data
flag_save_full=1
full_data_dir=data_dir


## Main

### Load in HERCULES database

In [4]:

#MAIN

##############################
#read in the database

Hdatabase=HERCULES_random_planet_database_1D()
Hdatabase.make_array(Hdir,Hname)
Hdatabase.initialize_interpolation([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],\
                                   [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]], flag_extrap=1)

#extract the latitudes for each Mu point
Nmu=Hdatabase.parr[0].Nmu
lat=np.arccos(Hdatabase.parr[0].layers[0].mu)*180/np.pi


Earth_correct_params_S3.20c
	 Earth_correct_params_S3.20c_AM3.3125
	 Earth_correct_params_S3.20c_AM3.30625
	 Earth_correct_params_S3.20c_AM3.325


### Read in the orbital data

In [5]:
if flag_data==0:
    dir=data_dir+'/rev8e3_for_simon'
    dirnames_const=['rev8a','rev8b', 'rev8c','']
elif flag_data==1:
    dir=data_dir+'/rev9_for_simon'
    dirnames_const=['rev9a','rev9b', 'rev9c','rev9e2']
    
Nruns_const=np.size(dirnames_const)

#########################
#read in the constant path
Ctime_const=[]
Ctime_high_const=[]
Ca_const=[]
Cob_const=[]
Comg_const=[]
CL_const=[]
Ce_const=[]
Ci_const=[]

for k in np.arange(np.size(dirnames_const)):
    
    ###Semi-major axis, e, i
    Cuk_f=open(dir+'/'+dirnames_const[k]+'/moon101.out', 'r')

    #extract the lines and strip them of any new lines etc.
    reader = csv.reader(Cuk_f, delimiter=" ", skipinitialspace=True)
    temp = list(reader)

    for i in np.arange(len(temp)):
        
        if (np.size(temp[i])>3):
            
            Ctime_const.append(temp[i][0]) #time
            Ca_const.append(temp[i][1]) #semi-major axis
            Ce_const.append(temp[i][2]) #eccentricity
            Ci_const.append(temp[i][3])
        

    ###obliquity
    Cuk_f=open(dir+'/'+dirnames_const[k]+'/pole.out', 'r')

    #extract the lines and strip them of any new lines etc.
    reader = csv.reader(Cuk_f, delimiter=" ", skipinitialspace=True)
    temp = list(reader)

    for i in np.arange(len(temp)):
        if np.size(temp[i])>4:
            Cob_const.append(temp[i][1]) #obliquity
            Comg_const.append(temp[i][3]) #rotation rate
            


#convert to useful type and unit
Ctime_const=np.asarray(Ctime_const, dtype=np.float64)
Ca_const=np.asarray(Ca_const, dtype=np.float64)
Cob_const=np.asarray(Cob_const, dtype=np.float64)
Ce_const=np.asarray(Ce_const, dtype=np.float64)
Ci_const=np.asarray(Ci_const, dtype=np.float64)
Comg_const=np.asarray(Comg_const, dtype=np.float64)/const.year
CL_const=Comg_const*C_Earth*MEarth*(REarth**2)

#remove any repeat values
temp=(Ctime_const[1:]-Ctime_const[:-1])
ind=np.where(temp==0.0)[0]
Ctime_const=np.delete(Ctime_const,ind+1)
Ca_const=np.delete(Ca_const,ind+1)
Cob_const=np.delete(Cob_const,ind+1)
Ce_const=np.delete(Ce_const,ind+1)
Ci_const=np.delete(Ci_const,ind+1)
Comg_const=np.delete(Comg_const,ind+1)
CL_const=np.delete(CL_const,ind+1)

#change names to be consistent with rest of program
Ctime=Ctime_const
Ca=Ca_const
Cob=Cob_const
Ce=Ce_const
Ci=Ci_const

Comg=Comg_const
CL=CL_const

    

print('done')

done


In [6]:
#define an output data file
if flag_overwrite==1:
    overwrite=''
else:
    overwrite='_recalc'
    
if flag_data==0:
    data_output_file_int=data_dir+'/Cuk21_surf_change_max_integrated_CukFig3'+overwrite+'.bin'
    data_output_file = full_data_dir+"/Cuk21_surf_change_CukFig3_delta"+overwrite+".h5"
elif flag_data==1:
    data_output_file_int=data_dir+'/Cuk21_surf_change_max_integrated_CukFig7'+overwrite+'.bin'
    data_output_file = full_data_dir+"/Cuk21_surf_change_CukFig7_delta"+overwrite+".h5"

### Loop over all calculated points, calculated integrated values, then save to file

For Cuk+ you also have the option to save the full arrays of surface deformation 

In [7]:
#Calcualte the change of surface at a selection of time points
print('begin')
tstart=tclock.time()

checkpoints=np.linspace(1,501,1001)
steps=np.arange(0,np.size(Ctime))
Nt=np.size(steps)
time=Ctime[steps]

if steps[0]>=np.size(Ctime):
    print('Too many!!!')
    print(ldkjfsl)

dLdt=gradient2(Ctime,CL)

if flag_save_full==1:

    #open the integrated data binary file
    dataf_int=open(data_output_file_int, "wb")

    
    # --------------------------------------------------
    # Names for hd5 file
    # --------------------------------------------------
    names = [
        "dl_lat", "dl_lon", "dA",
        "ddl_lat_dL", "ddl_lon_dL", "ddA_dL"
    ]

    #open the hd5 file
    with h5py.File(data_output_file, "w") as h5:

        #print the header info for the full output file
        h5.attrs["Nmu"] = Nmu
        h5.attrs["Nt"] = Nt
        h5.attrs["encoding"] = "delta-in-time, float32, gzip+shuffle" #the delta in time is vital

        # ----------------------------------------------
        # Time and dL/dt
        # ----------------------------------------------
        #time is encoded in a standard way
        time_ds = h5.create_dataset(
            "time", shape=(Nt,), dtype=np.float64
        )

        dLdt_ds = h5.create_dataset(
            "dL_dt",
            shape=(Nt,),
            dtype=np.float32,
            compression="gzip",
            compression_opts=4,
            shuffle=True
        )

        # ----------------------------------------------
        # Delta-encoded datasets
        # ----------------------------------------------
        #shape chanbes are encoded as deltas
        grp = h5.create_group("delta")

        dsets = {}
        for name in names:
            dsets[name] = grp.create_dataset(
                name,
                shape=(Nt, Nmu),
                dtype=np.float32,
                compression="gzip",
                compression_opts=4,
                shuffle=True,
                chunks=(32, Nmu)
            )

        # ----------------------------------------------
        # Streaming conversion
        # ----------------------------------------------
        prev = None
    
        count=-1
        for i in steps[0:]:
            count+=1
            #print(i, np.size(steps),count)
                
            if ((i-steps[0])*1.0/Nt*100)>checkpoints[0]:
                print((i-steps[0])*1.0/Nt*100, '%')
                checkpoints=checkpoints[1:]
            
            temp_data=Hdatabase.interp_database(CL[i],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],\
                                               [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]],flag_extrap=1)
        
            temp=np.asarray(temp_data[16])
        
            dl_lat=temp[:,0]
            dl_lon=temp[:,1]
            dA=temp[:,2]
            
            ddl_lat_dL=temp[:,3]
            ddl_lon_dL=temp[:,4]
            ddA_dL=temp[:,5]
            
            ddl_lat_dt=temp[:,3]*dLdt[i]
            ddl_lon_dt=temp[:,4]*dLdt[i]
            ddA_dt=temp[:,5]*dLdt[i]

            

            #print the full output to the hd5 file
            time_ds[i] =time[i]
            dLdt_ds[i]=dLdt[i]
            current = temp[:, :6].astype(np.float32)
        

            if prev is None:
                delta = current
            else:
                delta = current - prev

            for jj, name in enumerate(names):
                dsets[name][i, :] = delta[:, jj]

            prev = current

            ############
            #integrated deformation
        
            #longitudinal is easy. Accomodated around minor circle
            ddl_lon_dt_max=np.amax(ddl_lon_dt)*2*np.pi
            ddl_lon_dt_min=np.amin(ddl_lon_dt)*2*np.pi
        
        
            #latitudinal is harder. Need to integrate over surface
            temp=np.where(ddl_lat_dt<0)[0][-1]
        
            #print(lat[0:temp])
            ddl_lat_dt_min=2*integrate.trapz(ddl_lat_dt[0:temp]*np.pi/180.0,90.0-lat[0:temp])
            ddl_lat_dt_max=integrate.trapz(ddl_lat_dt[(temp+1):]*np.pi/180.0,90.0-lat[(temp+1):])
            
            #now print out these points to the file
            np.asarray(steps[i]).astype('float64').tofile(dataf_int)
            np.asarray(time[i]).astype('float64').tofile(dataf_int)
            np.asarray(ddl_lon_dt_max).astype('float64').tofile(dataf_int)
            np.asarray(ddl_lon_dt_min).astype('float64').tofile(dataf_int)
            np.asarray(ddl_lat_dt_min).astype('float64').tofile(dataf_int)
            np.asarray(ddl_lat_dt_max).astype('float64').tofile(dataf_int)
    
    dataf_int.close()

else: #just write the max integrated data

    #read in the file and loop over each data point
    dataf_int=open(data_output_file_int, "wb")

    
    count=-1
    for i in steps[0:]:
        count+=1
        #print(i, np.size(steps),count)
            
        if ((i-steps[0])*1.0/Nt*100)>checkpoints[0]:
            print((i-steps[0])*1.0/Nt*100, '%')
            checkpoints=checkpoints[1:]
        
        temp_data=Hdatabase.interp_database(CL[i],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],\
                                           [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]],flag_extrap=1)
    
        temp=np.asarray(temp_data[16])
    
        dl_lat=temp[:,0]
        dl_lon=temp[:,1]
        dA=temp[:,2]
        
        ddl_lat_dL=temp[:,3]
        ddl_lon_dL=temp[:,4]
        ddA_dL=temp[:,5]
        
        ddl_lat_dt=temp[:,3]*dLdt[i]
        ddl_lon_dt=temp[:,4]*dLdt[i]
        ddA_dt=temp[:,5]*dLdt[i]
    
        #longitudinal is easy. Accomodated around minor circle
        ddl_lon_dt_max=np.amax(ddl_lon_dt)*2*np.pi
        ddl_lon_dt_min=np.amin(ddl_lon_dt)*2*np.pi
    
    
        #latitudinal is harder. Need to integrate over surface
        temp=np.where(ddl_lat_dt<0)[0][-1]
    
        #print(lat[0:temp])
        ddl_lat_dt_min=2*integrate.trapz(ddl_lat_dt[0:temp]*np.pi/180.0,90.0-lat[0:temp])
        ddl_lat_dt_max=integrate.trapz(ddl_lat_dt[(temp+1):]*np.pi/180.0,90.0-lat[(temp+1):])
        
        #now print out these points to the file
        np.asarray(steps[i]).astype('float64').tofile(dataf_int)
        np.asarray(time[i]).astype('float64').tofile(dataf_int)
        np.asarray(ddl_lon_dt_max).astype('float64').tofile(dataf_int)
        np.asarray(ddl_lon_dt_min).astype('float64').tofile(dataf_int)
        np.asarray(ddl_lat_dt_min).astype('float64').tofile(dataf_int)
        np.asarray(ddl_lat_dt_max).astype('float64').tofile(dataf_int)
    
    dataf_int.close()


print('end')
print('time taken',tclock.time()-tstart)
print((tclock.time()-tstart)/(1.*count)*np.size(Ctime))
    

begin
1.000037431220133 %
1.5003680736646412 %
2.000074862440266 %
2.500405504884774 %
3.000112293660399 %
3.500442936104907 %
4.000149724880532 %
4.50048036732504 %
5.000187156100664 %
5.5005177985451725 %
6.000224587320798 %
6.500555229765307 %
7.000262018540932 %
7.500592660985439 %
8.000299449761064 %
8.50000623853669 %
9.000336880981196 %
9.500043669756822 %
10.000374312201329 %
10.500081100976955 %
11.000411743421463 %
11.500118532197087 %
12.000449174641597 %
12.50015596341722 %
13.000486605861731 %
13.500193394637353 %
14.000524037081863 %
14.500230825857487 %
15.000561468301996 %
15.500268257077622 %
16.000598899522128 %
16.500305688297754 %
17.00001247707338 %
17.500343119517886 %
18.000049908293512 %
18.50038055073802 %
19.000087339513644 %
19.50041798195815 %
20.000124770733777 %
20.500455413178283 %
21.00016220195391 %
21.500492844398416 %
22.00019963317404 %
22.50053027561855 %
23.000237064394174 %
23.500567706838684 %
24.00027449561431 %
24.500605138058816 %
25.000311926

### Compare to original data (if file not overwritten)

In [11]:
#If not overwritten can compare to previous binary file
if flag_overwrite==0:
    if flag_data==0:
        data_output_file_int_og=data_dir+'/Cuk21_surf_change_max_integrated_CukFig3.bin'
    elif flag_data==1:
        data_output_file_int_og=data_dir+'/Cuk21_surf_change_max_integrated_CukFig7.bin'
    
    #read in the max deformation data 
    dataf_int = open(data_output_file_int, "rb")
    dataf_int_og = open(data_output_file_int_og, "rb")
    
    #read in the file as one massive array
    data = np.fromfile(dataf_int, dtype=np.float64, count=-1)
    dataf_int.close()
    
    data_og = np.fromfile(dataf_int_og, dtype=np.float64, count=-1)
    dataf_int_og.close()
    
    print('Maximum abs and rel difference:', np.max(np.abs(data-data_og)),np.max(np.abs((data-data_og)/(data+1E-14))))



Maximum abs and rel difference: 1.3642420526593924e-12 4.030258517619553e-15


In [12]:
#pulls a few random time steps to check they are consistent from full data file. 
#Also checks the derived variables that were in the binary but not printed to hd5 and calculated later

# --------------------------------------------------
# User inputs
# --------------------------------------------------
#reproduced here to ensure consistency
if flag_data==0:
    data_output_file_og   = "../Lock_2026_SI_additional_data/Cuk21_surf_change_CukFig3_delta.h5"
elif flag_data==1:  
    data_output_file_og   = "../Lock_2026_SI_additional_data/Cuk21_surf_change_CukFig7_delta.h5"

# number of points to test
np.random.seed(0)
n_tests = 10

Ntest_max=10105

names = [
    "dl_lat", "dl_lon", "dA",
    "ddl_lat_dL", "ddl_lon_dL", "ddA_dL"
]

#fields that are in the binary file but not in the reduced delta format
derived_fields = ["dl_lat_dt", "dl_lon_dt", "dA_dt"]

# --------------------------------------------------
# Open files
# --------------------------------------------------
with h5py.File(data_output_file_og, "r") as h5_og, h5py.File(data_output_file, "r") as h5:

    Nt_og  = h5_og.attrs["Nt"]
    Nt  = h5.attrs["Nt"]

    print('Nt =', Nt_og, Nt)
    
    # --------------------------------------------------
    # Pick random timesteps
    # --------------------------------------------------
    test_steps = np.random.choice(np.min([Nt_og,Ntest_max]), size=n_tests, replace=False)
    print("Testing timesteps:", test_steps.tolist())

    # --------------------------------------------------
    # Precompute cumulative sums for delta fields
    # --------------------------------------------------
    cum_delta = {name: np.cumsum(h5["delta"][name][:Ntest_max, :], axis=0) for name in names}
    dL_dt = h5["dL_dt"][:]

    cum_delta_og = {name: np.cumsum(h5_og["delta"][name][:Ntest_max, :], axis=0) for name in names}
    dL_dt_og = h5_og["dL_dt"][:]

    # --------------------------------------------------
    # Compare times
    # --------------------------------------------------
    for j, t in enumerate(test_steps):
        h5_time = h5["time"][t]
        h5_time_og = h5["time"][t]
        abs_err = abs(h5_time_og - h5_time)
        rel_err = abs_err / max(abs(h5_time_og), 1e-12)
        print(f"\n--- Timestep {t} ---")
        print(f"time | abs err = {abs_err:.3e}, rel err = {rel_err:.3e}")

        # Compare stored fields
        for i, name in enumerate(names):
            aerr = np.max(np.abs(cum_delta_og[name] - cum_delta[name]))
            rerr = aerr / max(np.max(np.abs(cum_delta_og[name])), 1e-12)
            print(f"{name:<14} | max abs err = {aerr:.3e}, max rel err = {rerr:.3e}")

        # Compare derived _dt fields
        for i, field in enumerate(names[3:]):
            ref_dt = cum_delta_og[field] * h5_og["dL_dt"][j]            # binary _dt
            val_dt = cum_delta[field] * h5["dL_dt"][j]     # reconstructed from delta * dL_dt
            aerr = np.max(np.abs(ref_dt - val_dt))
            rerr = aerr / max(np.max(np.abs(ref_dt)), 1e-12)
            print(f"{field:<14} | max abs err = {aerr:.3e}, max rel err = {rerr:.3e}")

print("\nValidation complete.")


Nt = 160294 160294
Testing timesteps: [8201, 9338, 3456, 6035, 1421, 9347, 8944, 5488, 7835, 9883]

--- Timestep 8201 ---
time | abs err = 0.000e+00, rel err = 0.000e+00
dl_lat         | max abs err = 0.000e+00, max rel err = 0.000e+00
dl_lon         | max abs err = 0.000e+00, max rel err = 0.000e+00
dA             | max abs err = 0.000e+00, max rel err = 0.000e+00
ddl_lat_dL     | max abs err = 0.000e+00, max rel err = 0.000e+00
ddl_lon_dL     | max abs err = 0.000e+00, max rel err = 0.000e+00
ddA_dL         | max abs err = 0.000e+00, max rel err = 0.000e+00
ddl_lat_dL     | max abs err = 0.000e+00, max rel err = 0.000e+00
ddl_lon_dL     | max abs err = 0.000e+00, max rel err = 0.000e+00
ddA_dL         | max abs err = 0.000e+00, max rel err = 0.000e+00

--- Timestep 9338 ---
time | abs err = 0.000e+00, rel err = 0.000e+00
dl_lat         | max abs err = 0.000e+00, max rel err = 0.000e+00
dl_lon         | max abs err = 0.000e+00, max rel err = 0.000e+00
dA             | max abs err = 0.